# Tests de integración de `clean_trips()`

Notebook orientado a verificar el comportamiento end-to-end de OP-04 `Clean trips`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- y configuración básica de display.

### 0.1 Imports generales

In [1]:
from copy import deepcopy

import h3
import pandas as pd
from pandas.testing import assert_frame_equal

### 0.2 Imports del módulo 

In [2]:
from pylondrina.datasets import TripDataset
from pylondrina.reports import OperationReport
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.transforms.cleaning import CleanOptions, clean_trips

### 0.3 Helpers y Fixtures de testing reutilizables

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def make_valid_h3(lat: float, lon: float, res: int = 8) -> str:
    return h3.latlng_to_cell(lat, lon, res)


def make_clean_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    domain: DomainSpec | None = None,
    constraints: dict | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        domain=domain,
        constraints=constraints,
    )


BASE_CLEAN_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_clean_field("movement_id", "string", required=True),
        "user_id": make_clean_field("user_id", "string", required=True),
        "origin_longitude": make_clean_field("origin_longitude", "float"),
        "origin_latitude": make_clean_field("origin_latitude", "float"),
        "destination_longitude": make_clean_field("destination_longitude", "float"),
        "destination_latitude": make_clean_field("destination_latitude", "float"),
        "origin_h3_index": make_clean_field("origin_h3_index", "string"),
        "destination_h3_index": make_clean_field("destination_h3_index", "string"),
        "origin_time_utc": make_clean_field("origin_time_utc", "datetime"),
        "destination_time_utc": make_clean_field("destination_time_utc", "datetime"),
        "trip_id": make_clean_field("trip_id", "string"),
        "movement_seq": make_clean_field("movement_seq", "int"),
        "mode": make_clean_field(
            "mode",
            "categorical",
            domain=DomainSpec(values=["walk", "bus", "metro", "car", "unknown"], extendable=False),
        ),
        "purpose": make_clean_field(
            "purpose",
            "categorical",
            domain=DomainSpec(values=["home", "work", "study", "other", "unknown"], extendable=False),
        ),
        "day_type": make_clean_field(
            "day_type",
            "categorical",
            domain=DomainSpec(values=["weekday", "weekend", "holiday"], extendable=False),
        ),
        "user_gender": make_clean_field(
            "user_gender",
            "categorical",
            domain=DomainSpec(values=["female", "male", "other", "unknown"], extendable=False),
        ),
        "user_age_group": make_clean_field(
            "user_age_group",
            "categorical",
            domain=DomainSpec(values=["15-24", "25-34", "35-44", "45-54", "unknown"], extendable=False),
        ),
        "origin_municipality": make_clean_field("origin_municipality", "string"),
        "destination_municipality": make_clean_field("destination_municipality", "string"),
        "trip_weight": make_clean_field("trip_weight", "float"),
        "origin_time_local_hhmm": make_clean_field("origin_time_local_hhmm", "string"),
        "destination_time_local_hhmm": make_clean_field("destination_time_local_hhmm", "string"),
    },
    required=["movement_id", "user_id"],
    semantic_rules=None,
)


BASE_CLEAN_SCHEMA_EFFECTIVE = TripSchemaEffective(
    domains_effective={
        "mode": ["walk", "bus", "metro", "car", "unknown"],
        "purpose": ["home", "work", "study", "other", "unknown"],
        "day_type": ["weekday", "weekend", "holiday"],
        "user_gender": ["female", "male", "other", "unknown"],
        "user_age_group": ["15-24", "25-34", "35-44", "45-54", "unknown"],
    },
    temporal={"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
    fields_effective=list(BASE_CLEAN_SCHEMA.fields.keys()),
    dtype_effective={},
    overrides={},
)


def _canonical_rows():
    pts = [
        (-33.45, -70.66, -33.46, -70.67),
        (-33.46, -70.67, -33.47, -70.68),
        (-33.47, -70.68, -33.48, -70.69),
        (-33.48, -70.69, -33.49, -70.70),
        (-33.49, -70.70, -33.50, -70.71),
        (-33.50, -70.71, -33.51, -70.72),
    ]
    modes = ["walk", "bus", "metro", "car", "bus", "walk"]
    purposes = ["work", "study", "home", "work", "study", "home"]

    rows = []
    for i, (olat, olon, dlat, dlon) in enumerate(pts, start=1):
        rows.append(
            {
                "movement_id": f"m{i}",
                "user_id": f"u{i}",
                "origin_longitude": olon,
                "origin_latitude": olat,
                "destination_longitude": dlon,
                "destination_latitude": dlat,
                "origin_h3_index": make_valid_h3(olat, olon, 8),
                "destination_h3_index": make_valid_h3(dlat, dlon, 8),
                "origin_time_utc": f"2026-04-01T0{7+i}:00:00Z",
                "destination_time_utc": f"2026-04-01T0{7+i}:20:00Z",
                "trip_id": f"t{i}",
                "movement_seq": 0,
                "mode": modes[i - 1],
                "purpose": purposes[i - 1],
                "day_type": "weekday",
                "user_gender": "female" if i % 2 == 0 else "male",
                "user_age_group": "25-34" if i <= 3 else "35-44",
                "origin_municipality": "Santiago",
                "destination_municipality": "Providencia",
                "trip_weight": 1.0 + i / 10,
                "origin_time_local_hhmm": f"0{7+i}:00",
                "destination_time_local_hhmm": f"0{7+i}:20",
            }
        )
    return rows


def make_tripdataset_canonical_small() -> TripDataset:
    df = pd.DataFrame(_canonical_rows())
    return TripDataset(
        data=df.copy(),
        schema=BASE_CLEAN_SCHEMA,
        schema_version=BASE_CLEAN_SCHEMA.version,
        provenance={"source": {"name": "canonical_fixture"}},
        field_correspondence={},
        value_correspondence={},
        metadata={
            "dataset_id": "tripdataset_canonical_small",
            "events": [],
            "is_validated": False,
            "domains_effective": deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE.domains_effective),
            "temporal": {"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
        },
        schema_effective=deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE),
    )


def make_tripdataset_validated_small() -> TripDataset:
    df = pd.DataFrame(_canonical_rows())
    return TripDataset(
        data=df.copy(),
        schema=BASE_CLEAN_SCHEMA,
        schema_version=BASE_CLEAN_SCHEMA.version,
        provenance={"source": {"name": "validated_fixture"}},
        field_correspondence={"user_id": "id_persona", "mode": "modo"},
        value_correspondence={"mode": {"BUS": "bus", "METRO": "metro"}},
        metadata={
            "dataset_id": "tripdataset_validated_small",
            "events": [
                {
                    "op": "validate_trips",
                    "ts_utc": "2026-04-01T12:00:00Z",
                    "parameters": {"strict": False},
                    "summary": {"ok": True, "n_rows": 6},
                    "issues_summary": {
                        "counts": {"info": 0, "warning": 0, "error": 0},
                        "top_codes": [],
                    },
                }
            ],
            "is_validated": True,
            "domains_effective": deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE.domains_effective),
            "temporal": {"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
        },
        schema_effective=deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE),
    )


def make_tripdataset_dirty_small() -> TripDataset:
    rows = [
        # r0 - buena
        {
            "movement_id": "m1",
            "user_id": "u1",
            "origin_longitude": -70.66,
            "origin_latitude": -33.45,
            "destination_longitude": -70.67,
            "destination_latitude": -33.46,
            "origin_h3_index": make_valid_h3(-33.45, -70.66, 8),
            "destination_h3_index": make_valid_h3(-33.46, -70.67, 8),
            "origin_time_utc": "2026-04-01T08:00:00Z",
            "destination_time_utc": "2026-04-01T08:20:00Z",
            "trip_id": "t1",
            "movement_seq": 0,
            "mode": "walk",
            "purpose": "work",
            "day_type": "weekday",
            "user_gender": "male",
            "user_age_group": "25-34",
            "origin_municipality": "Santiago",
            "destination_municipality": "Providencia",
            "trip_weight": 1.0,
            "origin_time_local_hhmm": "08:00",
            "destination_time_local_hhmm": "08:20",
        },
        # r1 - duplicada de r0 según subset explícito
        {
            "movement_id": "m2",
            "user_id": "u1",
            "origin_longitude": -70.66,
            "origin_latitude": -33.45,
            "destination_longitude": -70.67,
            "destination_latitude": -33.46,
            "origin_h3_index": make_valid_h3(-33.45, -70.66, 8),
            "destination_h3_index": make_valid_h3(-33.46, -70.67, 8),
            "origin_time_utc": "2026-04-01T08:00:00Z",
            "destination_time_utc": "2026-04-01T08:20:00Z",
            "trip_id": "t2",
            "movement_seq": 0,
            "mode": "walk",
            "purpose": "work",
            "day_type": "weekday",
            "user_gender": "female",
            "user_age_group": "25-34",
            "origin_municipality": "Santiago",
            "destination_municipality": "Providencia",
            "trip_weight": 1.2,
            "origin_time_local_hhmm": "08:00",
            "destination_time_local_hhmm": "08:20",
        },
        # r2 - drop categórico por mode unknown
        {
            "movement_id": "m3",
            "user_id": "u2",
            "origin_longitude": -70.68,
            "origin_latitude": -33.47,
            "destination_longitude": -70.69,
            "destination_latitude": -33.48,
            "origin_h3_index": make_valid_h3(-33.47, -70.68, 8),
            "destination_h3_index": make_valid_h3(-33.48, -70.69, 8),
            "origin_time_utc": "2026-04-01T09:00:00Z",
            "destination_time_utc": "2026-04-01T09:25:00Z",
            "trip_id": "t3",
            "movement_seq": 0,
            "mode": "unknown",
            "purpose": "study",
            "day_type": "weekday",
            "user_gender": "female",
            "user_age_group": "15-24",
            "origin_municipality": "Ñuñoa",
            "destination_municipality": "Providencia",
            "trip_weight": 0.9,
            "origin_time_local_hhmm": "09:00",
            "destination_time_local_hhmm": "09:25",
        },
        # r3 - invalid_latlon por rango
        {
            "movement_id": "m4",
            "user_id": "u3",
            "origin_longitude": -70.70,
            "origin_latitude": 95.0,
            "destination_longitude": -70.71,
            "destination_latitude": -33.50,
            "origin_h3_index": make_valid_h3(-33.49, -70.70, 8),
            "destination_h3_index": make_valid_h3(-33.50, -70.71, 8),
            "origin_time_utc": "2026-04-01T10:00:00Z",
            "destination_time_utc": "2026-04-01T10:20:00Z",
            "trip_id": "t4",
            "movement_seq": 0,
            "mode": "bus",
            "purpose": "work",
            "day_type": "weekday",
            "user_gender": "male",
            "user_age_group": "35-44",
            "origin_municipality": "Las Condes",
            "destination_municipality": "Providencia",
            "trip_weight": 1.1,
            "origin_time_local_hhmm": "10:00",
            "destination_time_local_hhmm": "10:20",
        },
        # r4 - invalid_h3
        {
            "movement_id": "m5",
            "user_id": "u4",
            "origin_longitude": -70.72,
            "origin_latitude": -33.51,
            "destination_longitude": -70.73,
            "destination_latitude": -33.52,
            "origin_h3_index": "not_a_real_h3",
            "destination_h3_index": make_valid_h3(-33.52, -70.73, 8),
            "origin_time_utc": "2026-04-01T11:00:00Z",
            "destination_time_utc": "2026-04-01T11:20:00Z",
            "trip_id": "t5",
            "movement_seq": 0,
            "mode": "metro",
            "purpose": "study",
            "day_type": "weekday",
            "user_gender": "female",
            "user_age_group": "25-34",
            "origin_municipality": "Santiago",
            "destination_municipality": "Providencia",
            "trip_weight": 1.3,
            "origin_time_local_hhmm": "11:00",
            "destination_time_local_hhmm": "11:20",
        },
        # r5 - origin_after_destination
        {
            "movement_id": "m6",
            "user_id": "u5",
            "origin_longitude": -70.74,
            "origin_latitude": -33.53,
            "destination_longitude": -70.75,
            "destination_latitude": -33.54,
            "origin_h3_index": make_valid_h3(-33.53, -70.74, 8),
            "destination_h3_index": make_valid_h3(-33.54, -70.75, 8),
            "origin_time_utc": "2026-04-01T12:30:00Z",
            "destination_time_utc": "2026-04-01T12:00:00Z",
            "trip_id": "t6",
            "movement_seq": 0,
            "mode": "car",
            "purpose": "work",
            "day_type": "weekday",
            "user_gender": "male",
            "user_age_group": "45-54",
            "origin_municipality": "Maipú",
            "destination_municipality": "Santiago",
            "trip_weight": 1.0,
            "origin_time_local_hhmm": "12:30",
            "destination_time_local_hhmm": "12:00",
        },
        # r6 - null required
        {
            "movement_id": "m7",
            "user_id": None,
            "origin_longitude": -70.76,
            "origin_latitude": -33.55,
            "destination_longitude": -70.77,
            "destination_latitude": -33.56,
            "origin_h3_index": make_valid_h3(-33.55, -70.76, 8),
            "destination_h3_index": make_valid_h3(-33.56, -70.77, 8),
            "origin_time_utc": "2026-04-01T13:00:00Z",
            "destination_time_utc": "2026-04-01T13:20:00Z",
            "trip_id": "t7",
            "movement_seq": 0,
            "mode": "walk",
            "purpose": "home",
            "day_type": "weekday",
            "user_gender": "female",
            "user_age_group": "35-44",
            "origin_municipality": "Santiago",
            "destination_municipality": "Recoleta",
            "trip_weight": 0.8,
            "origin_time_local_hhmm": "13:00",
            "destination_time_local_hhmm": "13:20",
        },
        # r7 - nulls_fields sobre purpose
        {
            "movement_id": "m8",
            "user_id": "u7",
            "origin_longitude": -70.78,
            "origin_latitude": -33.57,
            "destination_longitude": -70.79,
            "destination_latitude": -33.58,
            "origin_h3_index": make_valid_h3(-33.57, -70.78, 8),
            "destination_h3_index": make_valid_h3(-33.58, -70.79, 8),
            "origin_time_utc": "2026-04-01T14:00:00Z",
            "destination_time_utc": "2026-04-01T14:25:00Z",
            "trip_id": "t8",
            "movement_seq": 0,
            "mode": "bus",
            "purpose": None,
            "day_type": "weekday",
            "user_gender": "male",
            "user_age_group": "25-34",
            "origin_municipality": "Puente Alto",
            "destination_municipality": "Santiago",
            "trip_weight": 1.4,
            "origin_time_local_hhmm": "14:00",
            "destination_time_local_hhmm": "14:25",
        },
        # r8 - buena
        {
            "movement_id": "m9",
            "user_id": "u8",
            "origin_longitude": -70.80,
            "origin_latitude": -33.59,
            "destination_longitude": -70.81,
            "destination_latitude": -33.60,
            "origin_h3_index": make_valid_h3(-33.59, -70.80, 8),
            "destination_h3_index": make_valid_h3(-33.60, -70.81, 8),
            "origin_time_utc": "2026-04-01T15:00:00Z",
            "destination_time_utc": "2026-04-01T15:20:00Z",
            "trip_id": "t9",
            "movement_seq": 0,
            "mode": "metro",
            "purpose": "study",
            "day_type": "weekday",
            "user_gender": "female",
            "user_age_group": "15-24",
            "origin_municipality": "Santiago",
            "destination_municipality": "Providencia",
            "trip_weight": 1.0,
            "origin_time_local_hhmm": "15:00",
            "destination_time_local_hhmm": "15:20",
        },
        # r9 - invalid_latlon por endpoint roto
        {
            "movement_id": "m10",
            "user_id": "u9",
            "origin_longitude": None,
            "origin_latitude": -33.61,
            "destination_longitude": -70.83,
            "destination_latitude": -33.62,
            "origin_h3_index": make_valid_h3(-33.61, -70.82, 8),
            "destination_h3_index": make_valid_h3(-33.62, -70.83, 8),
            "origin_time_utc": "2026-04-01T16:00:00Z",
            "destination_time_utc": "2026-04-01T16:20:00Z",
            "trip_id": "t10",
            "movement_seq": 0,
            "mode": "metro",
            "purpose": "work",
            "day_type": "weekday",
            "user_gender": "male",
            "user_age_group": "35-44",
            "origin_municipality": "Santiago",
            "destination_municipality": "Providencia",
            "trip_weight": 1.1,
            "origin_time_local_hhmm": "16:00",
            "destination_time_local_hhmm": "16:20",
        },
    ]

    df = pd.DataFrame(rows)

    return TripDataset(
        data=df.copy(),
        schema=BASE_CLEAN_SCHEMA,
        schema_version=BASE_CLEAN_SCHEMA.version,
        provenance={"source": {"name": "dirty_fixture"}},
        field_correspondence={"user_id": "id_persona", "mode": "modo"},
        value_correspondence={"mode": {"BUS": "bus", "METRO": "metro"}},
        metadata={
            "dataset_id": "tripdataset_dirty_small",
            "events": [
                {
                    "op": "validate_trips",
                    "ts_utc": "2026-04-01T12:00:00Z",
                    "parameters": {"strict": False},
                    "summary": {"ok": True, "n_rows": 10},
                    "issues_summary": {
                        "counts": {"info": 0, "warning": 0, "error": 0},
                        "top_codes": [],
                    },
                }
            ],
            "is_validated": True,
            "domains_effective": deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE.domains_effective),
            "temporal": {"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
        },
        schema_effective=deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE),
    )


show_ok("Helpers y fixtures reutilizables")

OK - Helpers y fixtures reutilizables


### 0.4 Configuración de display

In [4]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

### Test 1 - caso principal correcto: múltiples reglas sobre tripdataset_dirty_small

In [6]:
trips = make_tripdataset_dirty_small()

data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_nulls_in_required_fields=True,
        drop_rows_with_nulls_in_fields=["purpose"],
        drop_rows_with_invalid_latlon=True,
        drop_rows_with_invalid_h3=True,
        drop_rows_with_origin_after_destination=True,
        drop_duplicates=True,
        duplicates_subset=["user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"],
        drop_rows_by_categorical_values={"mode": ["unknown"]},
    ),
)

assert isinstance(cleaned, TripDataset)
assert isinstance(report, OperationReport)
assert report.ok is True

assert report.summary["rows_in"] == 10
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 8

assert report.summary["dropped_by_rule"]["nulls_required"] == 1
assert report.summary["dropped_by_rule"]["nulls_fields"] == 1
assert report.summary["dropped_by_rule"]["invalid_latlon"] == 2
assert report.summary["dropped_by_rule"]["invalid_h3"] == 1
assert report.summary["dropped_by_rule"]["origin_after_destination"] == 1
assert report.summary["dropped_by_rule"]["duplicates"] == 1
assert report.summary["dropped_by_rule"]["categorical_values"] == 1

assert list(cleaned.data["movement_id"]) == ["m1", "m9"]

assert cleaned.metadata["is_validated"] is True
assert len(cleaned.metadata["events"]) == len(events_before) + 1
assert cleaned.metadata["events"][:-1] == events_before

event = cleaned.metadata["events"][-1]
assert event["op"] == "clean_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before

display(trips.data)
display(cleaned.data)
display(report.summary)
display(report.parameters)
show_ok("Test 1 - caso principal correcto")

,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,mode,purpose,day_type,user_gender,user_age_group,origin_municipality,destination_municipality,trip_weight,origin_time_local_hhmm,destination_time_local_hhmm
0,m1,u1,-70.66,-33.45,-70.67,-33.46,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,t1,0,walk,work,weekday,male,25-34,Santiago,Providencia,1.0,08:00,08:20
1,m2,u1,-70.66,-33.45,-70.67,-33.46,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,t2,0,walk,work,weekday,female,25-34,Santiago,Providencia,1.2,08:00,08:20
2,m3,u2,-70.68,-33.47,-70.69,-33.48,88b2c555d1fffff,88b2c5558bfffff,2026-04-01T09:00:00Z,2026-04-01T09:25:00Z,t3,0,unknown,study,weekday,female,15-24,Ñuñoa,Providencia,0.9,09:00,09:25
3,m4,u3,-70.70,95.00,-70.71,-33.50,88b2c555b9fffff,88b2c555b5fffff,2026-04-01T10:00:00Z,2026-04-01T10:20:00Z,t4,0,bus,work,weekday,male,35-44,Las Condes,Providencia,1.1,10:00,10:20
4,m5,u4,-70.72,-33.51,-70.73,-33.52,not_a_real_h3,88b2c54747fffff,2026-04-01T11:00:00Z,2026-04-01T11:20:00Z,t5,0,metro,study,weekday,female,25-34,Santiago,Providencia,1.3,11:00,11:20
5,m6,u5,-70.74,-33.53,-70.75,-33.54,88b2c54705fffff,88b2c54721fffff,2026-04-01T12:30:00Z,2026-04-01T12:00:00Z,t6,0,car,work,weekday,male,45-54,Maipú,Santiago,1.0,12:30,12:00
6,m7,None,-70.76,-33.55,-70.77,-33.56,88b2c54727fffff,88b2c54569fffff,2026-04-01T13:00:00Z,2026-04-01T13:20:00Z,t7,0,walk,home,weekday,female,35-44,Santiago,Recoleta,0.8,13:00,13:20
7,m8,u7,-70.78,-33.57,-70.79,-33.58,88b2c54561fffff,88b2c54e5bfffff,2026-04-01T14:00:00Z,2026-04-01T14:25:00Z,t8,0,bus,None,weekday,male,25-34,Puente Alto,Santiago,1.4,14:00,14:25
8,m9,u8,-70.80,-33.59,-70.81,-33.60,88b2c54e57fffff,88b2c54e15fffff,2026-04-01T15:00:00Z,2026-04-01T15:20:00Z,t9,0,metro,study,weekday,female,15-24,Santiago,Providencia,1.0,15:00,15:20
9,m10,u9,NaN,-33.61,-70.83,-33.62,88b2c54e31fffff,88b2c54e37fffff,2026-04-01T16:00:00Z,2026-04-01T16:20:00Z,t10,0,metro,work,weekday,male,35-44,Santiago,Providencia,1.1,16:00,16:20


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,mode,purpose,day_type,user_gender,user_age_group,origin_municipality,destination_municipality,trip_weight,origin_time_local_hhmm,destination_time_local_hhmm
0,m1,u1,-70.66,-33.45,-70.67,-33.46,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,t1,0,walk,work,weekday,male,25-34,Santiago,Providencia,1.0,08:00,08:20
8,m9,u8,-70.80,-33.59,-70.81,-33.60,88b2c54e57fffff,88b2c54e15fffff,2026-04-01T15:00:00Z,2026-04-01T15:20:00Z,t9,0,metro,study,weekday,female,15-24,Santiago,Providencia,1.0,15:00,15:20


{'rows_in': 10,
 'rows_out': 2,
 'dropped_total': 8,
 'dropped_by_rule': {'nulls_required': 1,
  'nulls_fields': 1,
  'invalid_latlon': 2,
  'invalid_h3': 1,
  'origin_after_destination': 1,
  'duplicates': 1,
  'categorical_values': 1}}

{'drop_rows_with_nulls_in_required_fields': True,
 'drop_rows_with_nulls_in_fields': ['purpose'],
 'drop_rows_with_invalid_latlon': True,
 'drop_rows_with_invalid_h3': True,
 'drop_rows_with_origin_after_destination': True,
 'drop_duplicates': True,
 'duplicates_subset': ['user_id',
  'origin_time_utc',
  'origin_h3_index',
  'destination_h3_index'],
 'duplicates_subset_effective': ['user_id',
  'origin_time_utc',
  'origin_h3_index',
  'destination_h3_index'],
 'drop_rows_by_categorical_values': {'mode': ['unknown']}}

OK - Test 1 - caso principal correcto


### Test 2 - caso fatal/precondición: duplicates_subset explícito imposible

In [7]:
trips = make_tripdataset_validated_small()

data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    clean_trips(
        trips,
        options=CleanOptions(
            drop_duplicates=True,
            duplicates_subset=["campo_inexistente"],
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValueError)
assert getattr(raised, "code", None) == "CLN.CONFIG.INVALID_DUPLICATES_SUBSET"

assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before

show_ok("Test 2 - fatal/precondición")

OK - Test 2 - fatal/precondición


### Test 3 - caso degradado con warning: regla temporal no evaluable en Tier 2

In [10]:
trips = make_tripdataset_canonical_small()

# Se fuerza Tier 2 para que la regla temporal se omita con warning.
trips.metadata["temporal"] = {
    "tier": "tier_2",
    "fields_present": ["origin_time_local_hhmm", "destination_time_local_hhmm"],
}

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_origin_after_destination=True,
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "CLN.RULE.TEMPORAL_RULE_NOT_EVALUABLE" in codes
assert "CLN.NO_CHANGES.NO_RULES_ACTIVE" in codes

assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 6
assert report.summary["dropped_total"] == 0
assert report.summary["dropped_by_rule"]["origin_after_destination"] == 0

assert cleaned.metadata["is_validated"] is False
assert cleaned.metadata["events"][-1]["op"] == "clean_trips"

display(report.issues)
display(report.parameters)
show_ok("Test 3 - caso degradado con warning")

[Issue(level='warning', code='CLN.RULE.TEMPORAL_RULE_NOT_EVALUABLE', message="No se puede aplicar la regla temporal 'origin_after_destination': el dataset no es evaluable en Tier 1 o los tiempos no son comparables.", field=None, source_field=None, row_count=None, details={'rule': 'origin_after_destination', 'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'missing_fields': [], 'reason': 'temporal_rule_not_evaluable', 'action': 'rule_omitted'}),
 Issue(level='info', code='CLN.NO_CHANGES.NO_RULES_ACTIVE', message='No se aplicó ninguna regla de limpieza: no hay reglas activas o todas quedaron inactivas.', field=None, source_field=None, row_count=None, details={'rows_in': 6, 'rows_out': 6, 'dropped_total': 0, 'active_rules': [], 'omitted_rules': ['origin_after_destination'], 'reason': 'no_rules_active'})]

{'drop_rows_with_nulls_in_required_fields': False,
 'drop_rows_with_nulls_in_fields': None,
 'drop_rows_with_invalid_latlon': False,
 'drop_rows_with_invalid_h3': False,
 'drop_rows_with_origin_after_destination': True,
 'drop_duplicates': False,
 'duplicates_subset': None,
 'duplicates_subset_effective': None,
 'drop_rows_by_categorical_values': None}

OK - Test 3 - caso degradado con warning


### Test 4 - metadata / event / summary consistente y preservación contractual

In [12]:
trips = make_tripdataset_validated_small()

data_before = trips.data.copy(deep=True)
metadata_before = deepcopy(trips.metadata)
schema_effective_before = deepcopy(trips.schema_effective)
field_corr_before = deepcopy(trips.field_correspondence)
value_corr_before = deepcopy(trips.value_correspondence)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_by_categorical_values={"mode": ["metro"]},
    ),
)

assert report.ok is True
assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 5
assert report.summary["dropped_total"] == 1
assert report.summary["dropped_by_rule"]["categorical_values"] == 1

# Nuevo dataset, input intacto
assert cleaned is not trips
assert_frame_equal(trips.data, data_before)
assert trips.metadata == metadata_before

# Preservaciones obligatorias
assert cleaned.schema is trips.schema
assert cleaned.schema_version == trips.schema_version
assert cleaned.provenance == trips.provenance
assert cleaned.field_correspondence == field_corr_before
assert cleaned.value_correspondence == value_corr_before
assert cleaned.metadata["dataset_id"] == metadata_before["dataset_id"]
assert cleaned.metadata["is_validated"] is True
assert cleaned.metadata["domains_effective"] == metadata_before["domains_effective"]
assert "artifact_id" not in cleaned.metadata

# schema_effective preservado en contenido
assert cleaned.schema_effective.domains_effective == schema_effective_before.domains_effective
assert cleaned.schema_effective.temporal == schema_effective_before.temporal
assert cleaned.schema_effective.fields_effective == schema_effective_before.fields_effective

# Evento append-only solo en el resultado
assert len(trips.metadata["events"]) == 1
assert len(cleaned.metadata["events"]) == 2
assert cleaned.metadata["events"][:-1] == trips.metadata["events"]

event = cleaned.metadata["events"][-1]
assert event["op"] == "clean_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert set(event["issues_summary"].keys()) == {"counts", "top_codes"}

display(cleaned.metadata["events"])
show_ok("Test 4 - metadata / event / summary consistente")

[{'op': 'validate_trips',
  'ts_utc': '2026-04-01T12:00:00Z',
  'parameters': {'strict': False},
  'summary': {'ok': True, 'n_rows': 6},
  'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
   'top_codes': []}},
 {'op': 'clean_trips',
  'ts_utc': '2026-04-03T14:55:00.109817+00:00',
  'parameters': {'drop_rows_with_nulls_in_required_fields': False,
   'drop_rows_with_nulls_in_fields': None,
   'drop_rows_with_invalid_latlon': False,
   'drop_rows_with_invalid_h3': False,
   'drop_rows_with_origin_after_destination': False,
   'drop_duplicates': False,
   'duplicates_subset': None,
   'duplicates_subset_effective': None,
   'drop_rows_by_categorical_values': {'mode': ['metro']}},
  'summary': {'rows_in': 6,
   'rows_out': 5,
   'dropped_total': 1,
   'dropped_by_rule': {'nulls_required': 0,
    'nulls_fields': 0,
    'invalid_latlon': 0,
    'invalid_h3': 0,
    'origin_after_destination': 0,
    'duplicates': 0,
    'categorical_values': 1}},
  'issues_summary': {'count

OK - Test 4 - metadata / event / summary consistente


### Test 5 - política clave: duplicates_subset_effective por default

In [14]:
schema_for_default_dups = TripSchema(
    version="1.1",
    fields=deepcopy(BASE_CLEAN_SCHEMA.fields),
    required=["user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"],
    semantic_rules=None,
)

rows = _canonical_rows()
rows[1]["user_id"] = rows[0]["user_id"]
rows[1]["origin_time_utc"] = rows[0]["origin_time_utc"]
rows[1]["origin_h3_index"] = rows[0]["origin_h3_index"]
rows[1]["destination_h3_index"] = rows[0]["destination_h3_index"]

trips = TripDataset(
    data=pd.DataFrame(rows),
    schema=schema_for_default_dups,
    schema_version=schema_for_default_dups.version,
    provenance={"source": {"name": "default_dups_fixture"}},
    field_correspondence={},
    value_correspondence={},
    metadata={
        "dataset_id": "tripdataset_default_dups_small",
        "events": [],
        "is_validated": True,
        "domains_effective": deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE.domains_effective),
        "temporal": {"tier": "tier_1", "fields_present": ["origin_time_utc", "destination_time_utc"]},
    },
    schema_effective=deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE),
)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_duplicates=True,
        duplicates_subset=None,
    ),
)

assert report.ok is True
assert report.parameters["duplicates_subset"] is None
assert report.parameters["duplicates_subset_effective"] == [
    "user_id",
    "origin_time_utc",
    "origin_h3_index",
    "destination_h3_index",
]

assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 5
assert report.summary["dropped_total"] == 1
assert report.summary["dropped_by_rule"]["duplicates"] == 1

event = cleaned.metadata["events"][-1]
assert event["parameters"]["duplicates_subset_effective"] == [
    "user_id",
    "origin_time_utc",
    "origin_h3_index",
    "destination_h3_index",
]

display(trips.data)
display(cleaned.data)
show_ok("Test 5 - política clave de duplicados por default")

,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,mode,purpose,day_type,user_gender,user_age_group,origin_municipality,destination_municipality,trip_weight,origin_time_local_hhmm,destination_time_local_hhmm
0,m1,u1,-70.66,-33.45,-70.67,-33.46,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,t1,0,walk,work,weekday,male,25-34,Santiago,Providencia,1.1,08:00,08:20
1,m2,u1,-70.67,-33.46,-70.68,-33.47,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T09:20:00Z,t2,0,bus,study,weekday,female,25-34,Santiago,Providencia,1.2,09:00,09:20
2,m3,u3,-70.68,-33.47,-70.69,-33.48,88b2c555d1fffff,88b2c5558bfffff,2026-04-01T010:00:00Z,2026-04-01T010:20:00Z,t3,0,metro,home,weekday,male,25-34,Santiago,Providencia,1.3,010:00,010:20
3,m4,u4,-70.69,-33.48,-70.70,-33.49,88b2c5558bfffff,88b2c555b9fffff,2026-04-01T011:00:00Z,2026-04-01T011:20:00Z,t4,0,car,work,weekday,female,35-44,Santiago,Providencia,1.4,011:00,011:20
4,m5,u5,-70.70,-33.49,-70.71,-33.50,88b2c555b9fffff,88b2c555b5fffff,2026-04-01T012:00:00Z,2026-04-01T012:20:00Z,t5,0,bus,study,weekday,male,35-44,Santiago,Providencia,1.5,012:00,012:20
5,m6,u6,-70.71,-33.50,-70.72,-33.51,88b2c555b5fffff,88b2c5474bfffff,2026-04-01T013:00:00Z,2026-04-01T013:20:00Z,t6,0,walk,home,weekday,female,35-44,Santiago,Providencia,1.6,013:00,013:20


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,mode,purpose,day_type,user_gender,user_age_group,origin_municipality,destination_municipality,trip_weight,origin_time_local_hhmm,destination_time_local_hhmm
0,m1,u1,-70.66,-33.45,-70.67,-33.46,88b2c55417fffff,88b2c554e5fffff,2026-04-01T08:00:00Z,2026-04-01T08:20:00Z,t1,0,walk,work,weekday,male,25-34,Santiago,Providencia,1.1,08:00,08:20
2,m3,u3,-70.68,-33.47,-70.69,-33.48,88b2c555d1fffff,88b2c5558bfffff,2026-04-01T010:00:00Z,2026-04-01T010:20:00Z,t3,0,metro,home,weekday,male,25-34,Santiago,Providencia,1.3,010:00,010:20
3,m4,u4,-70.69,-33.48,-70.70,-33.49,88b2c5558bfffff,88b2c555b9fffff,2026-04-01T011:00:00Z,2026-04-01T011:20:00Z,t4,0,car,work,weekday,female,35-44,Santiago,Providencia,1.4,011:00,011:20
4,m5,u5,-70.70,-33.49,-70.71,-33.50,88b2c555b9fffff,88b2c555b5fffff,2026-04-01T012:00:00Z,2026-04-01T012:20:00Z,t5,0,bus,study,weekday,male,35-44,Santiago,Providencia,1.5,012:00,012:20
5,m6,u6,-70.71,-33.50,-70.72,-33.51,88b2c555b5fffff,88b2c5474bfffff,2026-04-01T013:00:00Z,2026-04-01T013:20:00Z,t6,0,walk,home,weekday,female,35-44,Santiago,Providencia,1.6,013:00,013:20


OK - Test 5 - política clave de duplicados por default


### Test 6 - sin cambios efectivos: reglas activas pero sin filas eliminadas

In [15]:
trips = make_tripdataset_canonical_small()

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_invalid_h3=True,
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "CLN.NO_CHANGES.NO_ROWS_DROPPED" in codes

assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 6
assert report.summary["dropped_total"] == 0
assert report.summary["dropped_by_rule"]["invalid_h3"] == 0

assert cleaned.metadata["is_validated"] is False
assert cleaned.metadata["events"][-1]["op"] == "clean_trips"

show_ok("Test 6 - sin cambios efectivos")

OK - Test 6 - sin cambios efectivos


### Test 7 - resultado vacío válido con warning no fatal

In [17]:
trips = make_tripdataset_canonical_small()

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_by_categorical_values={"mode": ["walk", "bus", "metro", "car"]},
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "CLN.RESULT.EMPTY_DATASET" in codes

assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 0
assert report.summary["dropped_total"] == 6
assert report.summary["dropped_by_rule"]["categorical_values"] == 6

assert cleaned.data.empty
assert cleaned.metadata["events"][-1]["op"] == "clean_trips"

display(report.issues)
show_ok("Test 7 - resultado vacío válido")

[Issue(level='warning', code='CLN.RESULT.EMPTY_DATASET', message='La limpieza produjo un dataset vacío.', field=None, source_field=None, row_count=None, details={'rows_in': 6, 'rows_out': 0, 'dropped_total': 6, 'active_rules': ['categorical_values'], 'dropped_by_rule': {'nulls_required': 0, 'nulls_fields': 0, 'invalid_latlon': 0, 'invalid_h3': 0, 'origin_after_destination': 0, 'duplicates': 0, 'categorical_values': 6}, 'reason': 'empty_dataset_after_clean'})]

OK - Test 7 - resultado vacío válido


### Test 8 - política semántica clave: OD parcial aceptado, endpoint roto rechazado

In [19]:
rows = [
    {
        "movement_id": "m_ok",
        "user_id": "u_ok",
        "origin_longitude": -70.66,
        "origin_latitude": -33.45,
        "destination_longitude": -70.67,
        "destination_latitude": -33.46,
        "trip_id": "t_ok",
        "movement_seq": 0,
        "mode": "walk",
        "purpose": "work",
    },
    {
        # OD parcial aceptado: origen completo válido, destino completamente ausente
        "movement_id": "m_partial_valid",
        "user_id": "u_partial",
        "origin_longitude": -70.68,
        "origin_latitude": -33.47,
        "destination_longitude": None,
        "destination_latitude": None,
        "trip_id": "t_partial",
        "movement_seq": 0,
        "mode": "bus",
        "purpose": "study",
    },
    {
        # Endpoint roto: lat presente, lon ausente
        "movement_id": "m_broken",
        "user_id": "u_broken",
        "origin_longitude": None,
        "origin_latitude": -33.49,
        "destination_longitude": -70.71,
        "destination_latitude": -33.50,
        "trip_id": "t_broken",
        "movement_seq": 0,
        "mode": "metro",
        "purpose": "work",
    },
]

df = pd.DataFrame(rows)
trips = TripDataset(
    data=df,
    schema=BASE_CLEAN_SCHEMA,
    schema_version=BASE_CLEAN_SCHEMA.version,
    provenance={"source": {"name": "od_partial_fixture"}},
    field_correspondence={},
    value_correspondence={},
    metadata={
        "dataset_id": "tripdataset_od_partial_small",
        "events": [],
        "is_validated": True,
        "domains_effective": deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE.domains_effective),
        "temporal": {"tier": "tier_3", "fields_present": []},
    },
    schema_effective=deepcopy(BASE_CLEAN_SCHEMA_EFFECTIVE),
)

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_with_invalid_latlon=True,
    ),
)

assert report.ok is True
assert report.summary["rows_in"] == 3
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 1
assert report.summary["dropped_by_rule"]["invalid_latlon"] == 1

assert list(cleaned.data["movement_id"]) == ["m_ok", "m_partial_valid"]

display(trips.data)
display(cleaned.data)
show_ok("Test 8 - OD parcial aceptado, endpoint roto rechazado")

,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_id,movement_seq,mode,purpose
0,m_ok,u_ok,-70.66,-33.45,-70.67,-33.46,t_ok,0,walk,work
1,m_partial_valid,u_partial,-70.68,-33.47,NaN,NaN,t_partial,0,bus,study
2,m_broken,u_broken,NaN,-33.49,-70.71,-33.50,t_broken,0,metro,work


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,trip_id,movement_seq,mode,purpose
0,m_ok,u_ok,-70.66,-33.45,-70.67,-33.46,t_ok,0,walk,work
1,m_partial_valid,u_partial,-70.68,-33.47,NaN,NaN,t_partial,0,bus,study


OK - Test 8 - OD parcial aceptado, endpoint roto rechazado


### Test 9 - warning por entrada categórica sobre campo no categórico

In [22]:
trips = make_tripdataset_canonical_small()

cleaned, report = clean_trips(
    trips,
    options=CleanOptions(
        drop_rows_by_categorical_values={"trip_weight": [1.1, 1.2]},
    ),
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "CLN.RULE.FIELD_NOT_CATEGORICAL" in codes
assert "CLN.NO_CHANGES.NO_RULES_ACTIVE" in codes

assert report.summary["rows_in"] == 6
assert report.summary["rows_out"] == 6
assert report.summary["dropped_total"] == 0
assert report.summary["dropped_by_rule"]["categorical_values"] == 0

display(report.issues)
show_ok("Test 9 - warning por campo no categórico")

[Issue(level='warning', code='CLN.RULE.FIELD_NOT_CATEGORICAL', message="No se puede aplicar drop por valores categóricos sobre 'trip_weight': el campo no es categórico ni interpretable como tal.", field='trip_weight', source_field=None, row_count=None, details={'rule': 'categorical_values', 'field': 'trip_weight', 'dtype_observed': 'float64', 'reason': 'field_not_categorical', 'action': 'rule_entry_omitted'}),
 Issue(level='info', code='CLN.NO_CHANGES.NO_RULES_ACTIVE', message='No se aplicó ninguna regla de limpieza: no hay reglas activas o todas quedaron inactivas.', field=None, source_field=None, row_count=None, details={'rows_in': 6, 'rows_out': 6, 'dropped_total': 0, 'active_rules': [], 'omitted_rules': ['categorical_values'], 'reason': 'no_rules_active'})]

OK - Test 9 - warning por campo no categórico


### Cobertura mínima de integración para OP-04 clean_trips:

1. Caso principal correcto ...................... OK
2. Fatal / precondición ......................... OK
3. Caso degradado con warning ................... OK
4. Metadata / event / summary ................... OK
5. Política clave: duplicates default ........... OK
6. Sin cambios efectivos ........................ OK
7. Resultado vacío válido ....................... OK
8. Política OD parcial / endpoint roto .......... OK
9. Warning por campo no categórico .............. OK